# 00 - Exploracion y validacion inicial del pipeline

Objetivo: explorar la disponibilidad, integridad y coherencia temporal de las tablas derivadas antes del analisis estadistico o de modelado.

Este notebook utiliza los scripts SQL de auditoria ubicados en `analysis/sql/qc/`. Las consultas no crean tablas; solo inspeccionan tablas existentes del dataset `mimic_analysis`.

## 1. Environment

Se cargan librerias, rutas del proyecto y parametros de conexion a BigQuery. El proyecto de facturacion puede definirse con la variable de entorno `BIGQUERY_PROJECT`; si no existe, se usa el proyecto observado en los SQL auditados.

In [ ]:
import os
from pathlib import Path

import pandas as pd
from pandas_gbq import read_gbq

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)

PROJECT_ROOT = Path.cwd()
ANALYSIS_DIR = PROJECT_ROOT / "analysis"
QC_SQL_DIR = ANALYSIS_DIR / "sql" / "qc"

BILLING_PROJECT_ID = os.environ.get("BIGQUERY_PROJECT", "strange-math-456415-c3")
DATASET_FQN = "strange-math-456415-c3.mimic_analysis"

print(f"Project root: {PROJECT_ROOT}")
print(f"QC SQL dir:   {QC_SQL_DIR}")
print(f"Billing project: {BILLING_PROJECT_ID}")
print(f"Dataset: {DATASET_FQN}")

## 2. Funciones auxiliares

`getSQL()` lee scripts versionados dentro del repositorio. `run_query()` ejecuta una consulta en BigQuery y devuelve un `DataFrame`, manteniendo el SQL trazable a archivo cuando se usa un script QC.

In [ ]:
def getSQL(relative_path: str) -> str:
    """Load a SQL file from the project root."""
    path = PROJECT_ROOT / relative_path
    if not path.exists():
        raise FileNotFoundError(f"SQL file not found: {path}")
    return path.read_text(encoding="utf-8")


def run_query(sql_or_path: str, project_id: str = BILLING_PROJECT_ID, **kwargs) -> pd.DataFrame:
    """Run SQL text or a .sql file path using pandas-gbq."""
    candidate_path = PROJECT_ROOT / sql_or_path
    sql = getSQL(sql_or_path) if candidate_path.exists() else sql_or_path
    return read_gbq(sql, project_id=project_id, dialect="standard", **kwargs)


QC_FILES = {
    "list_tables": "analysis/sql/qc/qc_00_list_tables.sql",
    "row_counts": "analysis/sql/qc/qc_01_row_counts.sql",
    "cohort_flow": "analysis/sql/qc/qc_02_cohort_flow.sql",
    "key_integrity": "analysis/sql/qc/qc_03_key_integrity.sql",
    "missingness": "analysis/sql/qc/qc_04_missingness.sql",
    "outcome_distribution": "analysis/sql/qc/qc_05_outcome_distribution.sql",
    "temporal_sanity": "analysis/sql/qc/qc_06_temporal_sanity.sql",
}

QC_FILES

## 3. Exploracion general

Se revisa disponibilidad de tablas, numero de filas, claves esperadas y duplicados. La unidad de analisis esperada cambia por etapa: una fila por estancia en tablas indice y una fila por estancia-dia en tablas longitudinales.

In [ ]:
tables = run_query(QC_FILES["list_tables"])
tables

In [ ]:
row_counts = run_query(QC_FILES["row_counts"])
row_counts.sort_values("table_name")

In [ ]:
key_integrity = run_query(QC_FILES["key_integrity"])
key_integrity.sort_values(["key_unique_flag", "table_name"])

Interpretacion esperada:

- `key_unique_flag = 1` indica que la clave declarada no presenta duplicados.
- Para tablas longitudinales, la clave esperada es `stay_id, day_idx`.
- Cualquier duplicado en la tabla final debe revisarse antes del analisis.

## 4. Cohorte

Se resume el flujo basico desde candidatos microbiologicos hasta la tabla longitudinal final. Este bloque cuantifica pacientes, ingresos, estancias y eventos microbiologicos por etapa.

In [ ]:
cohort_flow = run_query(QC_FILES["cohort_flow"])
cohort_flow

In [ ]:
cohort_summary = cohort_flow.assign(
    row_retention_from_previous=cohort_flow["n_rows"].div(cohort_flow["n_rows"].shift(1))
)
cohort_summary

## 5. Tabla final

Se evalua la tabla `longitudinal_cohort_model_ready`: numero de filas, numero de ventanas por paciente/estancia y distribucion de `day_idx`. Estas consultas usan solo la tabla final existente.

In [ ]:
final_table_summary_sql = f"""
SELECT
  COUNT(*) AS n_rows,
  COUNT(DISTINCT subject_id) AS n_subjects,
  COUNT(DISTINCT hadm_id) AS n_hadm,
  COUNT(DISTINCT stay_id) AS n_stays,
  COUNT(DISTINCT CONCAT(CAST(stay_id AS STRING), '|', CAST(day_idx AS STRING))) AS n_stay_days,
  MIN(day_idx) AS min_day_idx,
  MAX(day_idx) AS max_day_idx
FROM `{DATASET_FQN}.longitudinal_cohort_model_ready`
"""

final_table_summary = run_query(final_table_summary_sql)
final_table_summary

In [ ]:
windows_per_patient_sql = f"""
SELECT
  subject_id,
  COUNT(DISTINCT stay_id) AS n_stays,
  COUNT(*) AS n_windows,
  MIN(day_idx) AS min_day_idx,
  MAX(day_idx) AS max_day_idx
FROM `{DATASET_FQN}.longitudinal_cohort_model_ready`
GROUP BY subject_id
ORDER BY n_windows DESC
"""

windows_per_patient = run_query(windows_per_patient_sql)
windows_per_patient.describe(include="all")

In [ ]:
day_idx_distribution_sql = f"""
SELECT
  day_idx,
  COUNT(*) AS n_rows,
  COUNT(DISTINCT stay_id) AS n_stays,
  COUNT(DISTINCT subject_id) AS n_subjects
FROM `{DATASET_FQN}.longitudinal_cohort_model_ready`
GROUP BY day_idx
ORDER BY day_idx
"""

day_idx_distribution = run_query(day_idx_distribution_sql)
day_idx_distribution

## 6. Missing data

Se cuantifica el porcentaje de valores ausentes en variables clinicas clave, dominios de outcome y flags de mejoria.

In [ ]:
missingness = run_query(QC_FILES["missingness"])
missingness.sort_values("missing_fraction", ascending=False)

In [ ]:
missingness_by_table = (
    missingness
    .groupby("table_name", as_index=False)
    .agg(mean_missing_fraction=("missing_fraction", "mean"),
         max_missing_fraction=("missing_fraction", "max"),
         n_variables=("variable_name", "nunique"))
)
missingness_by_table

## 7. Outcome

Se inspecciona la distribucion de `improved_today`, `sustained_improvement` y los dominios que contribuyen a la definicion del outcome.

In [ ]:
outcome_distribution = run_query(QC_FILES["outcome_distribution"])
outcome_distribution

In [ ]:
outcome_focus = outcome_distribution[
    outcome_distribution["variable_name"].isin(["improved_today", "sustained_improvement"])
].copy()
outcome_focus

## 8. Checks temporales

Se validan reglas basicas de coherencia temporal: posicion de `true_t0` dentro de la UCI, ventana de +/- 48 h respecto al cultivo indice, y consistencia de `window_start`/`window_end`.

In [ ]:
temporal_sanity = run_query(QC_FILES["temporal_sanity"])
temporal_sanity.sort_values("n_violations", ascending=False)

## 9. Resumen de banderas de auditoria

Este bloque agrega indicadores simples para decidir si el pipeline esta listo para analisis. No sustituye revision clinica ni validacion metodologica.

In [ ]:
audit_flags = {
    "missing_expected_tables": int((tables["table_exists"] == 0).sum()),
    "empty_tables": int((row_counts["n_rows"] == 0).sum()),
    "tables_with_duplicate_keys": int((key_integrity["key_unique_flag"] == 0).sum()),
    "temporal_checks_with_violations": int((temporal_sanity["n_violations"] > 0).sum()),
}

pd.DataFrame([audit_flags])

## 10. Notas para reporte

Registrar antes del analisis:

- Tablas ausentes o vacias.
- Perdidas principales entre etapas de cohorte.
- Duplicados por clave esperada.
- Variables con missingness elevado.
- Distribucion de `improved_today` y `sustained_improvement`.
- Violaciones temporales en t0 o ventanas.

Si se detectan violaciones, corregir los SQL fuente o documentar la decision metodologica antes de construir notebooks analiticos.